# Lab 2: 제조 현장 특화 Tool 만들기

ReAct 에이전트의 힘은 **Tool**에서 나옵니다.
Tool은 실제 현장 시스템(MES, QMS, ERP)과 연결되는 인터페이스입니다.

**이 Lab에서 만들 Tool:**
1. `quality_inspector` — 품질 검사 결과 조회 (QMS 연동)
2. `anomaly_alerter` — 설비 이상 알림 전송 (알림 시스템 연동)
3. `inventory_checker` — 부품 재고 확인 (ERP 연동)
4. `work_order_creator` — 작업 지시서 생성 (MES 연동)

**학습 목표:**
- 현장 업무를 Tool 함수로 추상화하는 방법
- 여러 Tool을 조합하는 복합 Agent 실행
- Tool 설계 원칙 이해

In [ ]:
# tools/manufacturing_tools.py 임포트 (또는 직접 정의)
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..'))

# Tool 정의
def quality_inspector(product_id: str) -> str:
    """
    특정 제품의 품질 검사 결과 조회
    
    Args:
        product_id: 제품 ID (예: 'P001', 'P002')
    Returns:
        품질 검사 결과 문자열
    
    Example:
        >>> quality_inspector('P001')
        '제품 P001: 합격 (불량률: 0.02)'
    """
    mock_data = {
        "P001": {"status": "합격", "defect_rate": 0.02, "inspector": "AI-Vision-B1"},
        "P002": {"status": "불합격", "defect_rate": 0.15, "inspector": "AI-Vision-B1"},
        "P003": {"status": "합격", "defect_rate": 0.01, "inspector": "AI-Vision-B2"},
        "P004": {"status": "보류", "defect_rate": 0.08, "inspector": "AI-Vision-B1"},
    }
    result = mock_data.get(product_id, {"status": "미검사", "defect_rate": None, "inspector": "N/A"})
    status_icon = {"합격": "✅", "불합격": "❌", "보류": "⏳", "미검사": "❓"}.get(result['status'], "")
    return f"{status_icon} 제품 {product_id}: {result['status']} (불량률: {result.get('defect_rate', 'N/A')}, 검사기: {result['inspector']})"


def anomaly_alerter(location: str, severity: str = "warning") -> str:
    """
    설비 이상 알림 전송
    
    Args:
        location: 이상 발생 위치 (예: '라인 A 베어링', '압축기 3호')
        severity: 심각도 'critical'|'warning'|'info' (기본값: 'warning')
    Returns:
        알림 전송 결과 문자열
    
    Example:
        >>> anomaly_alerter('라인 A 베어링', 'critical')
        '🔴 [CRITICAL] 라인 A 베어링 이상 감지 → 정비팀 알림 전송 완료'
    """
    icons = {"critical": "🔴", "warning": "🟠", "info": "🟡"}
    icon = icons.get(severity, "⚪")
    recipients = {
        "critical": "정비팀장, 생산관리자, 공장장",
        "warning": "정비팀장, 생산관리자",
        "info": "정비팀"
    }.get(severity, "정비팀")
    return f"{icon} [{severity.upper()}] {location} 이상 감지 → 알림 전송 완료 (수신: {recipients})"


def inventory_checker(part_name: str) -> str:
    """
    부품 재고 확인
    
    Args:
        part_name: 부품명 (예: '베어링 6205', '오일씰 A형')
    Returns:
        재고 현황 문자열
    
    Example:
        >>> inventory_checker('베어링 6205')
        '베어링 6205: 12개 ✅ 충분'
    """
    inventory = {
        "베어링 6205": {"stock": 12, "min_stock": 5, "unit": "개", "location": "창고 A-3"},
        "오일씰 A형": {"stock": 3, "min_stock": 10, "unit": "개", "location": "창고 B-1"},
        "V벨트 B형": {"stock": 8, "min_stock": 5, "unit": "개", "location": "창고 A-5"},
        "필터 FE-100": {"stock": 0, "min_stock": 3, "unit": "개", "location": "입고 대기"},
    }
    item = inventory.get(part_name)
    if not item:
        return f"'{part_name}' 재고 정보 없음 (ERP 직접 확인 필요)"
    status = "✅ 충분" if item["stock"] >= item["min_stock"] else "⚠️ 부족 (발주 필요)"
    if item["stock"] == 0:
        status = "🚫 재고 없음 (긴급 발주)"
    return f"{part_name}: {item['stock']}{item['unit']} {status} | 위치: {item['location']}"


def work_order_creator(task: str, priority: str = "normal") -> str:
    """
    작업 지시서 생성 (MES 연동)
    
    Args:
        task: 작업 내용 (예: '라인 A 베어링 교체')
        priority: 우선순위 'urgent'|'normal'|'low' (기본값: 'normal')
    Returns:
        작업 지시서 ID 및 상세 정보
    
    Example:
        >>> work_order_creator('베어링 교체', 'urgent')
        '작업 지시서 생성: WO-03081423 | 내용: 베어링 교체 | 우선순위: urgent'
    """
    import datetime
    order_id = f"WO-{datetime.datetime.now().strftime('%m%d%H%M')}"
    priority_icon = {"urgent": "🔴", "normal": "🟡", "low": "🟢"}.get(priority, "⚪")
    estimated_hours = {"urgent": 2, "normal": 8, "low": 24}.get(priority, 8)
    return f"📋 작업 지시서 생성 완료: {order_id} | 내용: {task} | 우선순위: {priority_icon} {priority} | 예상 소요: {estimated_hours}h"


# Tool 목록 출력
tools = [quality_inspector, anomaly_alerter, inventory_checker, work_order_creator]
print(f"정의된 Tool 수: {len(tools)}개")
for tool in tools:
    print(f"  - {tool.__name__}: {tool.__doc__.strip().splitlines()[0]}")

In [ ]:
# 여러 Tool을 조합하는 복합 Agent 실행

class AdvancedManufacturingAgent:
    """복합 시나리오를 처리하는 제조 현장 ReAct 에이전트"""
    
    def __init__(self, tools):
        self.tools = {t.__name__: t for t in tools}
        self.thought_log = []
    
    def run(self, task: str, scenario: list):
        """
        scenario: [(thought, action_name, action_input), ...] 형태의 시나리오
        """
        print(f"🏭 시나리오 실행: {task}")
        print("=" * 60)
        
        for step, (thought, action_name, action_input) in enumerate(scenario, 1):
            print(f"\n[Step {step}]")
            print(f"  💭 Thought : {thought}")
            print(f"  ⚡ Action  : {action_name}({action_input!r})")
            
            # Tool 실행
            tool_fn = self.tools.get(action_name)
            if tool_fn:
                if isinstance(action_input, dict):
                    observation = tool_fn(**action_input)
                else:
                    observation = tool_fn(action_input)
            else:
                observation = f"⚠️ Tool '{action_name}'를 찾을 수 없습니다"
            
            print(f"  👁️  Observation: {observation}")
            
            self.thought_log.append({
                "step": step, "thought": thought,
                "action": action_name, "input": action_input,
                "observation": observation
            })
        
        print(f"\n{'=' * 60}")
        print(f"✅ 시나리오 완료 (총 {len(scenario)}단계 실행)")
        return self.thought_log


# 복합 시나리오: 품질 이상 → 재고 확인 → 작업 지시 → 알림
agent = AdvancedManufacturingAgent(tools)

scenario = [
    ("제품 P002의 품질 검사 결과를 먼저 확인해야 한다",
     "quality_inspector", "P002"),
    
    ("불합격이다. 원인이 베어링 마모일 가능성이 높다. 부품 재고를 확인한다",
     "inventory_checker", "베어링 6205"),
    
    ("재고가 충분하다. 즉시 베어링 교체 작업 지시서를 발행한다",
     "work_order_creator", {"task": "라인 B 베어링 교체 (품질 불합격 원인 조치)", "priority": "urgent"}),
    
    ("작업 지시서가 발행됐다. 관련 팀에 알림을 전송한다",
     "anomaly_alerter", {"location": "라인 B 품질 이상", "severity": "critical"}),
]

log = agent.run(
    "라인 B 불량품 발생 → 원인 파악 → 부품 확인 → 작업 지시 → 알림",
    scenario
)

## 핵심 정리

**Tool = 현장 시스템(MES, QMS)과 연결하는 인터페이스**

### Tool 설계 원칙
1. **단일 책임**: 하나의 Tool은 하나의 업무만 수행
2. **명확한 docstring**: 입력/출력 타입과 예시 필수
3. **Mock → 실제 연동**: 개발 단계에서는 Mock, 운영 시 실제 API 교체
4. **에러 처리**: 연동 실패 시 명확한 메시지 반환

### 실제 현장 연동 방법
```python
# Mock → 실제 MES API 연동 예시
def work_order_creator(task: str, priority: str = "normal") -> str:
    import requests
    response = requests.post(
        "http://mes.company.com/api/work-orders",
        json={"task": task, "priority": priority},
        headers={"Authorization": f"Bearer {os.getenv('MES_API_KEY')}"}
    )
    return response.json()["order_id"]
```

### 다음 단계
- LangSmith로 Agent 실행 과정 시각화
- 실제 LLM(GPT/Claude)으로 자동 Thought 생성
- 현장 API와 Tool 연동